## Building a RAG System with Langchain and FAISS

- Introduction to RAG (Retrieval-Augmented Generation) RAG combines the power of retrieval systems with generative AI models. Instead of relying solely on the model's training data, RAG:

1. Retrieves relevant documents from a knowledge base
2. Uses these documents as context for the LLM
3. Generates responses based on both the retrieved context and the model's knowledge.

FAISS:

- https//:github.com/facebookresearch/faiss

- FAISS is a library for efficient similarity search and clustering of dense vectors

- Key Advantage:
    1. Extremely fast similarity search
    2. Memory efficient
    3. Supports GPU acceleration
    4. Can handle millions of vectors

- How it works:
    - Indexes vectors for fast nearest neighbour search
    - Returns most similar vectors based on distance metrics

In [1]:
# load libraries
import os
from dotenv import load_dotenv
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Langchain core imports
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import (
    RunnablePassthrough
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage 


In [3]:
# Langchain specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Load environment varibles
load_dotenv()

True

#### Data Ingestion and Processing

In [4]:
sample_documents = [
    Document(
        page_content="""
        Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
        """,
        metadata={"source": "AI Introduction", "page": 1, "topic": "AI"}
    ),
    Document(
        page_content="""
        Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.
        """,
        metadata={"source": "ML Basics", "page": 1, "topic": "ML"}
    ),
    Document(
        page_content="""
        Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition.
        """,
        metadata={"source": "Deep Learning", "page": 1, "topic": "DL"}
    ),
    Document(
        page_content="""
        Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
        Applications include chatbots, translation, sentiment analysis, and text summarization.
        """,
        metadata={"source": "NLP Overview", "page": 1, "topic": "NLP"}
    )
]

print(sample_documents)

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='\n        Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.\n        '), Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='\n        Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.\n        '), Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='\n        Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolu

In [5]:
# text splitting
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=50,
    length_function=len,
    separators=[" "]
)

# Split the documents into chunks
chunks = text_splitter.split_documents(sample_documents)
print(chunks[0])
print(chunks[1])

page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.' metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}
page_content='Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.' metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}


In [6]:
print(f"Created {len(chunks)} chunks from {len(sample_documents)} documents")
print("\nExample chunk:")
print(f"Content: {chunks[0].page_content}")
print(f"Metadata: {chunks[0].metadata}")

Created 4 chunks from 4 documents

Example chunk:
Content: Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
Metadata: {'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}


In [7]:
# Load the embedding model
import os
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [8]:
# Initialize the OpenAI embeddings with the latest model
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=1536
)

# Example: create a embedding for a single text
sample_text = "What is machine learning"
sample_embedding = embeddings.embed_query(sample_text)
sample_embedding

[-0.005901336669921875,
 -0.005916595458984375,
 0.0005407333374023438,
 -0.0335693359375,
 0.0212249755859375,
 0.0221099853515625,
 -0.0008015632629394531,
 0.00928497314453125,
 -0.0226287841796875,
 0.03790283203125,
 0.01519012451171875,
 -0.03668212890625,
 -0.034027099609375,
 0.0015821456909179688,
 0.0267486572265625,
 0.015960693359375,
 0.0008187294006347656,
 -0.0308990478515625,
 0.02337646484375,
 0.00489044189453125,
 8.058547973632812e-05,
 0.032470703125,
 0.042236328125,
 -0.0293121337890625,
 0.0159759521484375,
 -0.0209503173828125,
 0.0266265869140625,
 0.01239776611328125,
 0.0074005126953125,
 -0.006023406982421875,
 -0.00939178466796875,
 -0.029052734375,
 -0.031341552734375,
 0.027557373046875,
 0.041748046875,
 0.0004277229309082031,
 -0.00807952880859375,
 -0.0106658935546875,
 -0.0555419921875,
 0.0026226043701171875,
 -0.056304931640625,
 -0.01629638671875,
 0.03021240234375,
 0.07611083984375,
 -0.0260009765625,
 -0.048828125,
 -0.036102294921875,
 -0.0353

In [9]:
texts = ["AI", "Machine Learning", "Deep Learning", "Neural Network"]
batch_embeddings = embeddings.embed_documents(texts)
print(batch_embeddings[0])

[-0.0081634521484375, -0.024658203125, 0.00293731689453125, 0.025115966796875, 0.006511688232421875, -0.028228759765625, -0.005035400390625, 0.0209197998046875, -0.036895751953125, 0.012786865234375, -0.003025054931640625, -0.0201263427734375, 0.0002956390380859375, -0.032745361328125, 0.0064697265625, -0.02532958984375, -0.031036376953125, -0.054351806640625, 0.032745361328125, -0.01837158203125, 0.01666259765625, 0.04827880859375, -0.0248870849609375, 0.0143585205078125, 0.0293731689453125, 0.00406646728515625, 0.00925445556640625, 0.01336669921875, 0.00254058837890625, -0.0225830078125, 0.032135009765625, -0.0279998779296875, 0.005359649658203125, -0.038177490234375, -0.0167388916015625, 0.0143280029296875, -0.03863525390625, -0.0103607177734375, -0.0105743408203125, -0.019195556640625, 0.03204345703125, 0.01459503173828125, -0.02154541015625, 0.0160675048828125, -0.0118865966796875, 0.0014524459838867188, -0.0048370361328125, -0.0335693359375, -0.0258331298828125, 0.045623779296875

In [10]:
print(batch_embeddings[1])

[-0.022064208984375, -0.0035247802734375, -0.0192108154296875, -0.0340576171875, 0.033782958984375, 0.008636474609375, 0.0014591217041015625, 0.0260009765625, -0.0413818359375, 0.042144775390625, -0.0007028579711914062, -0.037933349609375, -0.037628173828125, -0.0016155242919921875, 0.0160980224609375, 0.016510009765625, 0.0192413330078125, -0.0171051025390625, 0.0175323486328125, 0.0176849365234375, 0.02178955078125, 0.0242767333984375, 0.0199432373046875, -0.017333984375, 0.040008544921875, -0.020233154296875, 0.0291900634765625, 0.040863037109375, -0.007259368896484375, -0.027191162109375, -0.01491546630859375, -0.01454925537109375, -0.038055419921875, -0.0165557861328125, 0.0238800048828125, -0.0016469955444335938, -0.0030574798583984375, 0.0267791748046875, -0.049102783203125, 0.00862884521484375, -0.0289306640625, -0.0152130126953125, 0.01529693603515625, 0.0736083984375, -0.017547607421875, -0.01433563232421875, -0.0299072265625, -0.057403564453125, 0.0024166107177734375, 0.0680

In [11]:
# Compare Embedding using cosine similarity
def compare_embeddings(text1:str, text2:str):
    """Compare the semantic similarity of 2 texts using embeddings"""
    emb1 = np.array(embeddings.embed_query(text1))
    emb2 = np.array(embeddings.embed_query(text2))

    # Calculate the similarity score
    similarity = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
    return similarity

In [12]:
# Test semantic similarity
print("\nSemantic Similarity Examples:")
print(f"'AI' vs 'Artificial Intelligence': {compare_embeddings('AI', 'Artificial Intelligence'):.3f}")


Semantic Similarity Examples:
'AI' vs 'Artificial Intelligence': 0.563


In [13]:
print(f"'AI' vs 'Pizza': {compare_embeddings('AI', 'Pizza'):.3f}")

'AI' vs 'Pizza': 0.253


In [14]:
print(f"'Machine Learning' vs 'ML': {compare_embeddings('Machine Learning', 'ML'):.3f}")

'Machine Learning' vs 'ML': 0.461


#### Create FAISS vector store

In [15]:
vectorstore=FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)
print(f"Vector store created with {vectorstore.index.ntotal} vectors")

Vector store created with 4 vectors


In [16]:
vectorstore

In [17]:
## Save vector tore for later use
vectorstore.save_local("faiss_index")
print("Vector store saved to 'faiss_index' directory")

Vector store saved to 'faiss_index' directory


In [18]:
# Load vector store
loaded_vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

print(f"Loaded vector store contains {loaded_vectorstore.index.ntotal} vectors")

Loaded vector store contains 4 vectors


In [19]:
# Similarity Search 
query = "What is deep learning"

results = vectorstore.similarity_search(query, k=3)
print(results)

[Document(id='46188d63-cf2c-403f-8678-74154e031c69', metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recognition.'), Document(id='881a054d-4b40-4177-a861-aba363785e4c', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'), Document(id='17273251-33b4-42c5-92e0-1949828745ce', metadata={'source': 'NLP Overview', 'page': 1, 'topic': 'NLP'}, page_content='Natural Language Processing (NLP) is a branch of AI that helps computers understand human lang

In [20]:
print(f"Query: {query}\n")
print("Top 3 similar chunks:")

for i, doc in enumerate(results):
    print(f"\n{i+1}. Source: {doc.metadata['source']}")
    print(f"   Content: {doc.page_content[:200]}...")

Query: What is deep learning

Top 3 similar chunks:

1. Source: Deep Learning
   Content: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning ...

2. Source: ML Basics
   Content: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised...

3. Source: NLP Overview
   Content: Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
      ...


In [21]:
# Similarity Search with score
results_with_scores = vectorstore.similarity_search_with_score(query, k=3)

print("\n\nSimilarity search with scores:")
for doc, score in results_with_scores:
    print(f"\nScore: {score:.3f}")
    print(f"Source: {doc.metadata['source']}")
    print(f"Content preview: {doc.page_content[:100]}...")



Similarity search with scores:

Score: 0.555
Source: Deep Learning
Content preview: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses m...

Score: 1.207
Source: ML Basics
Content preview: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being...

Score: 1.273
Source: NLP Overview
Content preview: Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
...


In [22]:
chunks

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.'),
 Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'),
 Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recogn

In [23]:
# Search with metadata filtering
filter_dict = {"topic":"ML"}
filtered_results =vectorstore.similarity_search(
    query,
    k=3,
    filter=filter_dict
)
print(filtered_results)

[Document(id='881a054d-4b40-4177-a861-aba363785e4c', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.')]


In [24]:
len(filtered_results)

1

### Building RAG Chain with LCEL

In [25]:
# LLM GROQ LLM
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

# llm=init_chat_model(model="groq:gemma2-9b-it")  # model is decommissioned but serves as an example of how to initialize a chat model
llm = init_chat_model(model="groq:llama-3.3-70b-versatile")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F8F5327B10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F8F149BA10>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [26]:
llm.invoke("Hi")

AIMessage(content="It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 36, 'total_tokens': 59, 'completion_time': 0.048739424, 'completion_tokens_details': None, 'prompt_time': 0.003232643, 'prompt_tokens_details': None, 'queue_time': 0.169833777, 'total_time': 0.051972067}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d58d5-06f5-7f20-bc90-6f461a098f9e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 23, 'total_tokens': 59})

In [27]:
# 1. Simple RAG Chain with LCEL
simple_prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context:
Context: {context}

Question: {question}

Answer:""")

In [28]:
# Basic retriever
retriever=vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

In [29]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001F8F14773D0>, search_kwargs={'k': 3})

In [30]:
from typing import List

# Format documents for the prompt
def format_docs(docs: List[Document]) -> str:
    """Format documents for insertion into prompt"""
    formatted = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get('source', 'Unknown')
        formatted.append(f"Document {i+1} (Source: {source}):\n{doc.page_content}")
    return "\n\n".join(formatted)

In [31]:
simple_rag_chain=(
    {"context":retriever | format_docs,"question":RunnablePassthrough() }
    | simple_prompt
    | llm
    |StrOutputParser()
)

In [32]:
simple_rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001F8F14773D0>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\nContext: {context}\n\nQuestion: {question}\n\nAnswer:'), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F8F5327B10>, as

In [33]:
# Conversational RAG Chain
conversational_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Use the provided context to answer questions."),
    ("placeholder", "{chat_history}"),
    ("human", "Context: {context}\n\nQuestion: {input}"),
])

In [34]:
def create_conversational_rag():
    """Create a conversational RAG chain with memory"""
    return (
        RunnablePassthrough.assign(
            context=lambda x: format_docs(retriever.invoke(x["input"]))
        )
        | conversational_prompt
        | llm
        | StrOutputParser()
    )

conversational_rag = create_conversational_rag()

In [35]:
conversational_rag

RunnableAssign(mapper={
  context: RunnableLambda(lambda x: format_docs(retriever.invoke(x['input'])))
})
| ChatPromptTemplate(input_variables=['context', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag=

In [36]:
### streaming RAG chain
streaming_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | simple_prompt
    | llm
)
print("Modern RAG chains created successfully!")

Modern RAG chains created successfully!


- Available chains:
    - simple_rag_chain: Basic Q&A
    - conversational_rag: Maintains conversation history
    - streaming_rag_chain: Supports token streaming

In [37]:
# Test function for different chain types
def test_rag_chains(question: str):
    """Test all RAG chain variants"""
    print(f"Question: {question}")
    print("=" * 80)
    
    # 1. Simple RAG
    print("\n1. Simple RAG Chain:")
    answer = simple_rag_chain.invoke(question)
    print(f"Answer: {answer}")

    print("\n2. Streaming RAG:")
    print("Answer: ", end="", flush=True)
    for chunk in streaming_rag_chain.stream(question):
        print(chunk.content, end="", flush=True)
    print()

In [38]:
test_rag_chains("What is the difference between AI and machine learning")

Question: What is the difference between AI and machine learning

1. Simple RAG Chain:
Answer: According to the provided context, Artificial Intelligence (AI) is the simulation of human intelligence in machines, whereas Machine Learning is a subset of AI that enables systems to learn from data instead of being explicitly programmed. In other words, AI is a broader concept that encompasses Machine Learning, which is a specific technique used to achieve AI's goals.

2. Streaming RAG:
Answer: According to the context, the difference between AI and Machine Learning is that Artificial Intelligence (AI) is the simulation of human intelligence in machines, designed to think like humans and mimic their actions. Machine Learning, on the other hand, is a subset of AI that enables systems to learn from data, finding patterns in data instead of being explicitly programmed.


In [39]:
# Test with multiple questions
test_questions = [
    "What is the difference between AI and Machine Learning?",
    "Explain deep learning in simple terms",
    "How does NLP work?"
]

for question in test_questions:
    print("\n" + "=" * 80 + "\n")
    test_rag_chains(question)



Question: What is the difference between AI and Machine Learning?

1. Simple RAG Chain:
Answer: Based on the provided context, Artificial Intelligence (AI) is the simulation of human intelligence in machines, while Machine Learning (ML) is a subset of AI that enables systems to learn from data. In other words, AI is a broader concept that encompasses ML, which is a specific approach to achieving AI through learning from data.

2. Streaming RAG:
Answer: Based on the provided context, Artificial Intelligence (AI) is the simulation of human intelligence in machines, while Machine Learning (ML) is a subset of AI that enables systems to learn from data. In other words, AI is a broader concept that encompasses ML, which is a specific technique used to achieve AI's goals by finding patterns in data.


Question: Explain deep learning in simple terms

1. Simple RAG Chain:
Answer: Deep learning is a type of machine learning that uses multiple layers to help computers understand and learn from 

In [40]:
# Conversational example
print("\n3. Conversational RAG Example:")
chat_history = []

# First question
q1 = "What is machine learning?"
a1 = conversational_rag.invoke({
    "input": q1,
    "chat_history": chat_history
})

print(f"Q1: {q1}")
print(f"A1: {a1}")


3. Conversational RAG Example:
Q1: What is machine learning?
A1: Machine Learning is a subset of Artificial Intelligence (AI) that enables systems to learn from data, rather than being explicitly programmed. It involves finding patterns in data through various algorithms, including supervised, unsupervised, and reinforcement learning.


In [41]:
# Update history
chat_history.extend([
    HumanMessage(content=q1),
    AIMessage(content=a1)
])

In [42]:
# Follow-up question
q2 = "How is it different from traditional programming?"
a2 = conversational_rag.invoke({
    "input": q2,
    "chat_history": chat_history
})
print(f"\nQ2: {q2}")
print(f"A2: {a2}")


Q2: How is it different from traditional programming?
A2: According to Document 2 (Source: ML Basics), Machine Learning is different from traditional programming in that it enables systems to learn from data instead of being explicitly programmed. This means that with traditional programming, a computer is given a set of rules and instructions to follow, whereas with machine learning, the system is able to find patterns in data and make decisions or predictions on its own.
